## Camada Gold —-> Análises

Este notebook gera três visões analíticas a partir dos dados enriquecidos da camada Silver:
faturamento mensal, ranking de vendedores por receita e top 3 produtos por categoria,
utilizando agregações e Functions.

**Fonte:** `/Volumes/dados/default/desafio_03/silver/pedidos_enriquecidos`  
**Destino:** `/Volumes/dados/default/desafio_03/gold/`  
**Visões:** faturamento_mensal, ranking_vendedores, top_produtos_categoria

In [0]:
# 03_gold.py
# Camada Gold — Visões analíticas

from pyspark.sql import functions as F
from pyspark.sql.window import Window

SILVER_PATH = "/Volumes/dados/default/desafio_03/silver"
GOLD_PATH   = "/Volumes/dados/default/desafio_03/gold"

# Leitura da Silver
df = spark.read.format("delta").load(f"{SILVER_PATH}/pedidos_enriquecidos")

# Gold 1 — Faturamento Mensal 

faturamento = df \
    .withColumn("ano",  F.year("data_pedido")) \
    .withColumn("mes",  F.month("data_pedido")) \
    .groupBy("ano", "mes") \
    .agg(
        F.countDistinct("id_pedido").alias("total_pedidos"),
        F.round(F.sum("valor_total_item"), 2).alias("receita_total")
    ) \
    .orderBy("ano", "mes")

faturamento.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/faturamento_mensal")
print(" Gold 1 — Faturamento Mensal ")
display(faturamento)

#  Gold 2 — Ranking de Vendedores 

vendedores = df \
    .groupBy("nome_vendedor", "regiao") \
    .agg(
        F.countDistinct("id_pedido").alias("total_pedidos"),
        F.round(F.sum("valor_total_item"), 2).alias("receita_gerada"),
        F.round(F.avg("valor_total_item"), 2).alias("ticket_medio")
    ) \
    .orderBy(F.desc("receita_gerada"))

vendedores.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/ranking_vendedores")
print(" Gold 2 — Ranking de Vendedores ")
display(vendedores)

#  Gold 3 — Top Produtos por Categoria 

produtos = df \
    .groupBy("categoria", "nome_produto") \
    .agg(
        F.sum("quantidade").alias("quantidade_vendida"),
        F.round(F.sum("valor_total_item"), 2).alias("receita_produto")
    )

window_spec = Window.partitionBy("categoria").orderBy(F.desc("receita_produto"))

top_produtos = produtos \
    .withColumn("ranking", F.rank().over(window_spec)) \
    .filter(F.col("ranking") <= 3)

top_produtos.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/top_produtos_categoria")
print(" Gold 3 — Top Produtos por Categoria ")
display(top_produtos)

print("Gold concluído")